# MNIST adatbázis

A feladatban a MNIST adatbázist fogjuk használni, amely egy klasszikus gépi tanulási adathalmaz kézírásos számjegyek felismerésére.

Az adatbázis 28×28 pixeles, szürkeárnyalatos képeket tartalmaz, amelyek kézzel írt számjegyeket ábrázolnak 0-tól 9-ig.

Az adathalmaz két részre van osztva:
- tanító halmaz: 60 000 kép
- teszt halmaz: 10 000 kép

Minden képhez tartozik egy címke, amely megadja, hogy a képen melyik számjegy (0–9) látható.

A feladat ezek alapján egy olyan neurális háló felépítése, amely képes a képeken szereplő számjegyek helyes felismerésére.

# 1. feladat – Adatok betöltése és előkészítése

A feladatban a Fashion MNIST adatbázist kell beolvasni, amely a Keras beépített adathalmaza.

Az adat a `keras.datasets.mnist.load_data` modulból érhető el, ezért nincs szükség külön letöltésre vagy külső fájlokra.

A betöltés után el kell végezni a képek normalizálását, vagyis a pixelértékeket 0–255 tartományból 0–1 közé kell skálázni.

In [1]:
from tensorflow import keras

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

x_train = x_train / 255.0
x_test = x_test / 255.0

C:\Users\USER\anaconda3\envs\prog1\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


# 2. feladat – Modell felépítése

A feladatban egy neurális háló modellt kell készíteni a képek osztályozására.

A modellnek tartalmaznia kell:
- egy Flatten réteget, amely a 28×28-as képeket egy vektorrá alakítja
- legalább egy Dense (teljesen összekötött) réteget ReLU aktivációval
- egy kimeneti Dense réteget 10 neuronnal, Softmax aktivációval (10 osztály miatt)

A modell szerkezete szabadon bővíthető (pl. több réteg, dropout), de a fenti elemek kötelezőek.

In [2]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),

    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(64, activation='relu'),

    layers.Dense(10, activation='softmax')
])

C:\Users\USER\anaconda3\envs\prog1\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


# 3. feladat – A modell fordítása (kompilálása)

A modell tanításának megkezdése előtt szükséges megadni, hogy hogyan tanuljon a háló.

Ebben a lépésben ki kell választani:
- az optimalizálási módszert (optimizer)
- a veszteségfüggvényt (loss function)
- valamint a kiértékelési metrikát (metric)

Mivel ez egy többosztályos osztályozási feladat (10 osztály), ennek megfelelő beállításokat kell használni.

In [3]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 4. feladat – Tanítás, kiértékelés és a modell mentése

A feladat utolsó lépésében a modellt be kell tanítani, majd ki kell értékelni a teszt adatokon.

Ezután a betanított modellt el kell menteni, hogy később újra felhasználható legyen anélkül, hogy újratanítanánk.

A tanítás során figyeld a tanulási folyamatot (loss és accuracy értékek), valamint a validációs teljesítményt is.

A modell mentése egy fájlba történik, amelyet később vissza lehet tölteni és újra használni. A modell elmentéséről én már gondoskodtam, a te feladatod a modell betanítása ebben a részben.

In [4]:
model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.1
)

loss, acc = model.evaluate(x_test, y_test)
print("Accuracy:", acc)

Epoch 1/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.8869 - loss: 0.3729 - val_accuracy: 0.9668 - val_loss: 0.1089
Epoch 2/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9475 - loss: 0.1701 - val_accuracy: 0.9738 - val_loss: 0.0876
Epoch 3/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9601 - loss: 0.1295 - val_accuracy: 0.9760 - val_loss: 0.0796
Epoch 4/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9653 - loss: 0.1126 - val_accuracy: 0.9800 - val_loss: 0.0672
Epoch 5/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9681 - loss: 0.0991 - val_accuracy: 0.9795 - val_loss: 0.0720
Epoch 6/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9727 - loss: 0.0873 - val_accuracy: 0.9802 - val_loss: 0.0688
Epoch 7/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9739 - loss: 0.0783 - val_accuracy: 0.9817 - val_loss: 0.0652
Epoch 8/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9767 - loss: 0.0726 - val_accuracy: 0.

In [5]:
MODEL_PATH = "mnist_simple.keras"

model.save(MODEL_PATH)